<a href="https://colab.research.google.com/github/prpNn0p/machinetranslation/blob/main/simpletransformers_mt_en_th.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install simpletransformers

In [ ]:
pip install -U --no-cache-dir gdown --pre

In [ ]:
!gdown --id 18J1a6YEIUFWUHtW9T6uS0h7gBJOZqxOS

In [ ]:
%mv /content/Official_Dataset_Machine\ Translation.zip /content/Official_Dataset_Machine_Translation.zip

In [ ]:
!unzip -qq /content/Official_Dataset_Machine_Translation.zip

In [ ]:
import os
import pandas as pd

# Functions

In [ ]:
def extract_data(file):
  f = open(file)
  x = f.read()

  output = x.split("\n")
  return output

In [ ]:
def convert_df_to_format(input_df):
  data = []
  for row, series in input_df.iterrows():
    data.append(["translate thai to english", series['th_text'], series['en_text']])
    # data.append(["translate english to thai", series['en_text'], series['th_text']])
  output_df = pd.DataFrame(data, columns=["prefix", "input_text", "target_text"])
  return output_df

# Train

## Train Datasets

In [ ]:
path_en = '/content/TrainSet/LST_ENTH680K/lowercase/lang.en'
path_th = '/content/TrainSet/LST_ENTH680K/lowercase/lang.th'
lower_case_df = pd.DataFrame(extract_data(path_en), columns = ['en_text'])
lower_case_df['th_text'] = extract_data(path_th)
lower_case_df = lower_case_df[:681539]
lower_case_df

,en_text,th_text
0,thank you but i don't have enough time,ขอบคุณ แต่ ฉัน ไม่ มี เวลา พอ
1,"oh, you know, when i get up i usually do my st...","โอ้ , คุณ รู้ , เมื่อ ฉัน ตื่น ฉัน จะ ออกไป เด..."
2,the duty of a policeman is to safeguard people .,หน้าที่ ของ ตำรวจ คือ การ ปกป้อง ประชาชน
3,it's female vanity.,มัน เป็นความ โอหัง ของ เพศ หญิง .
4,please keep a sharp lookout .,โปรด คอย ดู อย่า เผลอ
...,...,...
681534,"when wijai talked to me on the phone , his voi...",ตอน วิจัย พูด โทรศัพท์ กับ ดิฉัน เสียง เขา แตก...
681535,the bus terminal is on the west side of the ci...,สถานี รถ โดยสาร ทาง ตะวันตก ของ เมือง
681536,encapsulate,ลักษณะ ภาย นอก
681537,"death frightens me , specifically my own death .",ความ ตาย ทำ ให้ ผม กลัว โดย เฉพาะอย่างยิ่ง การ...


In [ ]:
frames = [lower_case_df] #Ubuntu_df , qed_df, tanzil_df, tatoeba_df,  Bible_df, GNOME_df, JW300_df, opensub_df
all_df = pd.concat(frames)

In [ ]:
# Test convert function
all_formatted = convert_df_to_format(all_df)
all_formatted

,prefix,input_text,target_text
0,translate thai to english,ขอบคุณ แต่ ฉัน ไม่ มี เวลา พอ,thank you but i don't have enough time
1,translate thai to english,"โอ้ , คุณ รู้ , เมื่อ ฉัน ตื่น ฉัน จะ ออกไป เด...","oh, you know, when i get up i usually do my st..."
2,translate thai to english,หน้าที่ ของ ตำรวจ คือ การ ปกป้อง ประชาชน,the duty of a policeman is to safeguard people .
3,translate thai to english,มัน เป็นความ โอหัง ของ เพศ หญิง .,it's female vanity.
4,translate thai to english,โปรด คอย ดู อย่า เผลอ,please keep a sharp lookout .
...,...,...,...
681534,translate thai to english,ตอน วิจัย พูด โทรศัพท์ กับ ดิฉัน เสียง เขา แตก...,"when wijai talked to me on the phone , his voi..."
681535,translate thai to english,สถานี รถ โดยสาร ทาง ตะวันตก ของ เมือง,the bus terminal is on the west side of the ci...
681536,translate thai to english,ลักษณะ ภาย นอก,encapsulate
681537,translate thai to english,ความ ตาย ทำ ให้ ผม กลัว โดย เฉพาะอย่างยิ่ง การ...,"death frightens me , specifically my own death ."


## Train Model

In [ ]:
import logging
import pandas as pd
from simpletransformers.t5 import T5Model, T5Args


logging.basicConfig(level=logging.INFO)
transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)

ModuleNotFoundError: ignored

In [ ]:
model_args = T5Args()
model_args.max_seq_length = 96
model_args.train_batch_size = 8
#model_args.eval_batch_size = 4
model_args.num_train_epochs = 5
model_args.evaluate_during_training = False
#model_args.evaluate_during_training_steps = 200
model_args.use_multiprocessing = False
model_args.fp16 = False
model_args.save_steps = -1
model_args.save_eval_checkpoints = False
model_args.no_cache = True
model_args.reprocess_input_data = True
model_args.overwrite_output_dir = True
model_args.preprocess_inputs = False
model_args.num_return_sequences = 1
#model_args.wandb_project = "MT5 Sinhala-English Translation"

model = T5Model("mt5", "/content/checkpoint-85193-epoch-1/", args=model_args)

In [ ]:
# Train the model
%%timeit
model.train_model(all_formatted)  #(train_df, eval_data=eval_df)

INFO:simpletransformers.t5.t5_utils: Creating features from dataset file at cache_dir/


  0%|          | 0/681539 [00:00<?, ?it/s]

/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:3524: FutureWarning: 
`prepare_seq2seq_batch` is deprecated and will be removed in version 5 of HuggingFace Transformers. Use the regular
`__call__` method to prepare your inputs and the tokenizer under the `as_target_tokenizer` context manager to prepare
your targets.

Here is a short example:

model_inputs = tokenizer(src_texts, ...)
with tokenizer.as_target_tokenizer():
    labels = tokenizer(tgt_texts, ...)
model_inputs["labels"] = labels["input_ids"]

See the documentation of your specific tokenizer for more details on the specific arguments to the tokenizer of choice.
For a more complete example, see the implementation of `prepare_seq2seq_batch`.

  warnings.warn(formatted_warning, FutureWarning)
INFO:simpletransformers.t5.t5_model: Training started


Epoch:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:simpletransformers.t5.t5_model:   Starting fine-tuning.


Running Epoch 0 of 5:   0%|          | 0/85193 [00:00<?, ?it/s]

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

INFO:root:
Unfortunately, your original traceback can not be constructed.



Traceback (most recent call last):
  File "/usr/local/lib/python3.7/dist-packages/IPython/core/interactiveshell.py", line 3524, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_1044/3847668455.py", line 1, in <module>
    get_ipython().run_cell_magic('timeit', '', 'model.train_model(all_formatted)  #(train_df, eval_data=eval_df)\n')
  File "/usr/local/lib/python3.7/dist-packages/IPython/core/interactiveshell.py", line 2462, in run_cell_magic
    result = fn(*args, **kwargs)
  File "<decorator-gen-53>", line 2, in timeit
  File "/usr/local/lib/python3.7/dist-packages/IPython/core/magic.py", line 187, in <lambda>
    call = lambda f, *a, **k: f(*a, **k)
  File "/usr/local/lib/python3.7/dist-packages/IPython/core/magics/execution.py", line 1180, in timeit
    time_number = timer.timeit(number)
  File "/usr/local/lib/python3.7/dist-packages/IPython/core/magics/execution.py", line 169, in timeit
    timing = self.inner(it, self.timer)
  File "<magic-t

TypeError: ignored

In [ ]:
!zip -r /content/best_model_en_th_lst.zip /content/outputs/best_model/

In [ ]:
from google.colab import drive
drive.mount('/content/mydrive/')

## Download Trained Model

In [ ]:
!unzip -qq /content/en-th_T5_model.zip

In [ ]:
!gdown --id 104_LbQFgqNM7RRjwUFr1IfMxeXFHDGNe

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=104_LbQFgqNM7RRjwUFr1IfMxeXFHDGNe
To: /content/pytorch_model.bin
100% 2.33G/2.33G [00:14<00:00, 158MB/s]


In [ ]:
!mv /content/pytorch_model.bin /content/checkpoint-85193-epoch-1/pytorch_model.bin

In [ ]:
!pip install sacrebleu

In [ ]:
import logging
import sacrebleu
import pandas as pd
from simpletransformers.t5 import T5Model, T5Args


logging.basicConfig(level=logging.INFO)
transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)


model_args = T5Args()
model_args.max_length = 96
model_args.length_penalty = 1
model_args.num_beams = 10

model = T5Model("mt5", "/content/checkpoint-85193-epoch-1/", args=model_args)

# Validation

In [ ]:
!gdown --id 18J1a6YEIUFWUHtW9T6uS0h7gBJOZqxOS

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=18J1a6YEIUFWUHtW9T6uS0h7gBJOZqxOS
To: /content/Official_TrainningDataset_Machine Translation.zip
100% 238M/238M [00:09<00:00, 24.9MB/s]


In [ ]:
%mv /content/Official_TrainningDataset_Machine\ Translation.zip /content/Official_Dataset_Machine_Translation.zip

In [ ]:
!unzip -qq /content/Official_Dataset_Machine_Translation.zip

In [ ]:
path_en = '/content/ValidationSet/Valid1+2/dev.en'
path_th = '/content/ValidationSet/Valid1+2/dev.th'
valid1_df = pd.DataFrame(extract_data(path_en), columns = ['en_text'])
valid1_df['th_text'] = extract_data(path_th)
valid1_df

,en_text,th_text
0,It has been confirmed that eight thoroughbred ...,ได้ เป็น ที่ ยืนยัน แน่นอน แล้ว ว่า ม้า แข่ง พ...
1,"Randwick has been locked down, and is expected...",สนาม แข่ง ม้า แร ็นด์ วิค ถูก ปิด และ คาด ว่า ...
2,It is expected that the virulent flu will affe...,คาด ว่า ไข้ หวัด ใหญ่ ร้ายแรง นี้ จะ กระทบ ต่อ...
3,NSW Minister for Primary Industries said the f...,รัฐมนตรี อุตสาหกรรม พื้นฐาน นอง รัฐNSW กล่าว ว...
4,The cases are the first infections of race hor...,เหตุการณ์ นี้ คือ การ ติด เชื้อ ครั้ง แรก ใน ม...
...,...,...
1014,They are reported to have verified her identit...,มี รายงาน แจ้ง ว่า พวก เขา ทราบ ตัว ตน ของ เธอ...
1015,TMZ.com reports that police were checking to s...,TMZ.com รายงาน ว่า ทาง ตำรวจ ได้ ตรวจสอบ ใบ อน...
1016,Spears was not charged with anything and despi...,Spears ไม่ ถูก ปรับ ใน ข้อ หา ใด ๆ และ แม้ ว่า...
1017,"""Britney Spears was part of the group, but was...",SaraFaden โฆษก หญิง ของ กรมตำรวจ LosAngeles กล...


In [ ]:
path_en = '/content/ValidationSet/Valid1+2/dev2.en'
path_th = '/content/ValidationSet/Valid1+2/dev2.th'
valid2_df = pd.DataFrame(extract_data(path_en), columns = ['en_text'])
valid2_df['th_text'] = extract_data(path_th)
valid2_df

,en_text,th_text
0,Did you get that thesis from the Dude?,คุณได้ข้อเสนอนี้ จากเพื่อนคนนั้นเหรอ?
1,Police! Get your hands up!,มันเป็นช่วงนาทีที่ ดึงคุณออกจากตัวคุณเอง
2,"We should eat. You hungry, buddy? Yeah.",กินกันเถอะ หิวแล้วใช่มั้ย
3,I'd watch myself if I were you. (Coughs) Ok.,ถ้าฉันเป็นนาย จะคอยระวังตัวไว้ โอเค ที่นี่ล่ะ
4,Anyhow ... it appears you left a favorable imp...,ยังไงก็ตามดูเหมือนว่าคุณทำให้ที่ประชุมวันนี้ ป...
...,...,...
1996,We must close the gate.,เราจะต้องปิดประตู.
1997,You said I was free to come and go.,คุณพูดเองว่าฉันมีสิทธิ์ที่จะไปไหนมาไหนก็ได้
1998,"Well, we're on pins and needles.",เหน็บกินแล้วนะ
1999,"Kjetil, we have to get up. We have to get up.",เชทิล ตื่นเถอะ


In [ ]:
frames = [valid1_df]  # valid2_df # valid1_df
valid_df = pd.concat(frames, ignore_index=True)
valid_df = valid_df[:1018]
valid_df

,en_text,th_text
0,It has been confirmed that eight thoroughbred ...,ได้ เป็น ที่ ยืนยัน แน่นอน แล้ว ว่า ม้า แข่ง พ...
1,"Randwick has been locked down, and is expected...",สนาม แข่ง ม้า แร ็นด์ วิค ถูก ปิด และ คาด ว่า ...
2,It is expected that the virulent flu will affe...,คาด ว่า ไข้ หวัด ใหญ่ ร้ายแรง นี้ จะ กระทบ ต่อ...
3,NSW Minister for Primary Industries said the f...,รัฐมนตรี อุตสาหกรรม พื้นฐาน นอง รัฐNSW กล่าว ว...
4,The cases are the first infections of race hor...,เหตุการณ์ นี้ คือ การ ติด เชื้อ ครั้ง แรก ใน ม...
...,...,...
1013,Police also say that at least one of cars foll...,ตำรวจ ยัง เสริม ว่า มี รถ อย่าง น้อย หนึ่ง คัน...
1014,They are reported to have verified her identit...,มี รายงาน แจ้ง ว่า พวก เขา ทราบ ตัว ตน ของ เธอ...
1015,TMZ.com reports that police were checking to s...,TMZ.com รายงาน ว่า ทาง ตำรวจ ได้ ตรวจสอบ ใบ อน...
1016,Spears was not charged with anything and despi...,Spears ไม่ ถูก ปรับ ใน ข้อ หา ใด ๆ และ แม้ ว่า...


In [ ]:
thai_truth = valid_df['th_text'].tolist()
to_thai = valid_df['en_text'].tolist()

In [ ]:
thai_truth

['ได้ เป็น ที่ ยืนยัน แน่นอน แล้ว ว่า ม้า แข่ง พันธุ์ ดี แปด ตัว ที่ สนาม แข่งแรนด์วิค ใน เมืงซิดนีย์ ได้ ติด เชื้อ ไข้ หวัด ใหญ่ ม้า',
 'สนาม แข่ง ม้า แร ็นด์ วิค ถูก ปิด และ คาด ว่า จะ ยังคง ปิด อยู่ ต่อ ไป อีก ถึง 2 เดือน',
 'คาด ว่า ไข้ หวัด ใหญ่ ร้ายแรง นี้ จะ กระทบ ต่อ ม้า ส่วน ใหญ่ จำนวน 700 ตัว ใน คอก ม้า ที่ ที่ แรนด์วิค',
 'รัฐมนตรี อุตสาหกรรม พื้นฐาน นอง รัฐNSW กล่าว ว่า จะ กัก บริเวณ ไว้ 30 วัน หลัง จาก ที่ ได้ พบ สัญญาณ สุดท้าย ของ ไข้ หวัด',
 'เหตุการณ์ นี้ คือ การ ติด เชื้อ ครั้ง แรก ใน ม้า แข่ง ทั้ง ที่ ได้ มี การ ติด เชื้อ ใน ม้า สำหรับ กิจกรรม สันทนาการ อื่น ๆ หลาย สิบ ตัว ทั่ว รัฐNWS และ รัฐควีนส์แลนด์',
 'ไข้ หวัด นี้ มี โอกาส ติด เชื้อ โดย ทาง สัมผัส ได้ สูง มาก แต่ ไม่ สามารถ แพร่สามนุษย์',
 'การ ปิด แข่ง ม้า ของ ประเทศ ทำ ให้ ต้อง สิ้นเปลือง ค่า ใช้จ่าย ของ อุตสาหกรรม หลาย สิบ ล้าน ดอลลาร์ ทุก วัน',
 'ประธาน กรรมการ บริหาร ของ ม้า แข่ง ประจำ รัฐNSWPeterV \' Landys กล่าว ว่า การ แข่ง ม้า ถูก ขัดขวาง ตั้งแต่ มี การ ห้าม เคลื่อนย้าย ม้า เมื่อ สุด สัปดาห์ ที่ ผ่าน มา

In [ ]:
thai_preds = model.predict(to_thai)

Generating outputs:   0%|          | 0/128 [00:00<?, ?it/s]

/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:3524: FutureWarning: 
`prepare_seq2seq_batch` is deprecated and will be removed in version 5 of HuggingFace Transformers. Use the regular
`__call__` method to prepare your inputs and the tokenizer under the `as_target_tokenizer` context manager to prepare
your targets.

Here is a short example:

model_inputs = tokenizer(src_texts, ...)
with tokenizer.as_target_tokenizer():
    labels = tokenizer(tgt_texts, ...)
model_inputs["labels"] = labels["input_ids"]

See the documentation of your specific tokenizer for more details on the specific arguments to the tokenizer of choice.
For a more complete example, see the implementation of `prepare_seq2seq_batch`.

  warnings.warn(formatted_warning, FutureWarning)


Decoding outputs:   0%|          | 0/1018 [00:00<?, ?it/s]

In [ ]:
thai_preds

['มี การ ยืนยัน ว่า ม้า วิ่ง แข่ง ที่ ได้ รับ การ อบรม อย่าง ละเอียด แปด ตัว ที่ สนาม แข่ง วิ่ง แข่ง ที่ วิ่ง แข่ง วิ่ง แข่ง ที่ สนาม แข่ง วิ่ง แข่ง ที่ สนาม แข่ง วิ่ง แข่ง ที่ สนาม แข่ง วิ่ง แข่ง ที่ สนาม แข่ง วิ่ง แข่ง ที่ วิ่ง มา วิ่ง แข่ง ที่ สนาม แข่ง',
 'คาดหวัง ว่า รัน ด ์ วิค ถูก ล็อก ไว้, และ ถูก คาดหวัง ที่ จะ อยู่ จนถึง สอง เดือน.',
 'คาดหวัง ว่า ไข้ พุพอง จะ ส่งผล กระทบ ส่วนใหญ่ ของ ม้า 7 0 0 ตัว ที่ ยืน อยู่ ที่ รัน วิค.',
 'รัฐมนตรี ว่าการ เกษตรศาสตร์ กล่าว ว่า สิ่ง อํานวย ความ สะดวก จะ ถูก กักตัว จนถึง สามสิบ วัน หลัง จาก สัญญาณ สุดท้าย ของ ไข้หวัด.',
 'กรณี นี้ คือ การ ติด เชื้อ ครั้ง แรก ของ ม้า แข่ง ม้า, ถึงแม้ ว่าจะ ติด เชื้อ ม้า หลาย สิบ ตัว ของ ม้า สาธารณะ ข้าม เอ็น ส ์ แลนด์.',
 'ไข้หวัด เป็น พาหะ ติด เชื้อ อย่าง สูง แต่ ไม่ สามารถ ถ่ายทอด ไปยัง มนุษย์ ได้.',
 'ทุกวัน นี้ การ ปิด การ แข่งขัน วิ่ง แข่ง ของ ประเทศ กําลัง ส่งผล ให้ อุตสาหกรรม สิบล้าน ดอลล่าร์ ทุกวัน.',
 'หัวหน้า ของ หัวหน้า ของ การ แข่ง ม้า N S, พี เ ตอ ร ์ V in g s กล่าว ขณะ ที่ การ แข่งขัน ของ ม้า 

In [ ]:
# def post_process(pred):
#   post_th_pred = []
#   for sentence in pred:
#     post_th_pred = string.replace("&apos;", "'")

#   return post_th_pred

In [ ]:
# english_preds_new = post_process(english_preds)

In [ ]:
eng_th_bleu = sacrebleu.corpus_bleu(thai_preds, thai_truth)
print("English to Thai: ", eng_th_bleu.score)

English to Thai:  0.1010877860789219


# Submission Testset

In [ ]:
!gdown --id 1khuxHPjTTR5d2tL356i8qd-M8JFK2gFu

/usr/local/lib/python3.7/dist-packages/gdown/cli.py:131: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  category=FutureWarning,
Downloading...
From: https://drive.google.com/uc?id=1khuxHPjTTR5d2tL356i8qd-M8JFK2gFu
To: /content/testset_v1_3fnd5.zip
100% 106k/106k [00:00<00:00, 15.6MB/s]


In [ ]:
!unzip -qq /content/testset_v1_3fnd5.zip

replace TestSet_ForSuperAI_Release01/test.en? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [ ]:
path_en = '/content/TestSet_ForSuperAI_Release01/test.en'
# path_th = '/content/TestSet_ForSuperAI_Release01/test.th'
test_df = pd.DataFrame(extract_data(path_en), columns = ['en_text'])
# test_df['th_text'] = extract_data(path_th)
test_df = test_df[:1520]
test_df

,en_text
0,What! on fire!
1,what is your hobby?
2,This is a tragic love story.
3,Is this road safe?
4,Please turn in your assignments on time.
...,...
1515,He criticised the outgoing head of the U.N.'s ...
1516,"""In the past, the United States government sto..."
1517,ExxonMobil wishes to assure members of the pub...
1518,The inquest was opened at the request of famil...


In [ ]:
# thai_truth = valid_df['th_text'].tolist()
to_thai = test_df['en_text'].tolist()

In [ ]:
thai_preds = model.predict(to_thai)

Generating outputs:   0%|          | 0/190 [00:00<?, ?it/s]

/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:3524: FutureWarning: 
`prepare_seq2seq_batch` is deprecated and will be removed in version 5 of HuggingFace Transformers. Use the regular
`__call__` method to prepare your inputs and the tokenizer under the `as_target_tokenizer` context manager to prepare
your targets.

Here is a short example:

model_inputs = tokenizer(src_texts, ...)
with tokenizer.as_target_tokenizer():
    labels = tokenizer(tgt_texts, ...)
model_inputs["labels"] = labels["input_ids"]

See the documentation of your specific tokenizer for more details on the specific arguments to the tokenizer of choice.
For a more complete example, see the implementation of `prepare_seq2seq_batch`.

  warnings.warn(formatted_warning, FutureWarning)


Decoding outputs:   0%|          | 0/1520 [00:00<?, ?it/s]

In [ ]:
thai_preds

['เกิด อะไร ขึ้น บน ไฟ ไหม้?',
 'งาน อดิเรก ของ คุณ คือ อะไร?',
 'นี่ เป็น เรื่อง รัก ที่ เศร้าสร้อย.',
 'ถนน นี้ ปลอดภัย ไหม?',
 'กรุณา โอน งาน ของ คุณ ตรง เวลา.',
 'ช่าง น่า เสียดาย อะไร อย่างนี้?',
 'แม่ เอา ตุ๊กตา ใหม่ ให้ ฉัน, และ ฉัน ชอบ มัน.',
 'สามี ของ ฉัน ทํา ให้ ฉัน สบายใจ เสมอ.',
 'เขา สนิท กับ พ่อ ของ เขา มาก.',
 'ปักกิ่ง เป็น เมืองหลวง ของ จีน.',
 'ขอบคุณ ที่ ช่วย ฉัน. บี - ยินดี ต้อนรับ.',
 'ฉัน ไม่ ชอบ ว่ายน้ํา.',
 'ใบ หน้า ของ เขา เป็น สีขาว เหมือน กระดาษ.',
 'เรา ต้องการ ความ ช่วยเหลือ ที่ เชี่ยวชาญ.',
 'เรา มา แข่งขัน กัน ดู ว่า ใคร วิ่ง ไป ที่ ต้น ใหญ่ ก่อน.',
 'อย่า ทํา ให้ ฉัน ยุ่งยาก.',
 'เข็มกลัด สูง กว่า นี้ บ้าง ไหม?',
 'เขา เอา กุหลาบ สีแดง ให้ ฉัน หนึ่ง ช่อ.',
 'เที่ยวบิน ทีจี สาม ศูนย์ มา ถึง เวลา กี่ โมง?',
 'เรา จะ พบ คุณ พรุ่ง นี้ เวลา แปดโมง ครึ่ง ที่ ประตู โรงเรียน.',
 'ไม่ มี ใคร ใน สํานัก งาน.',
 'กระเป๋า ของ เขา ติด อยู่ บน โต๊ะ.',
 'ฉัน ไม่ ต้องการ กิน, ฉัน อิ่ม.',
 'คุณ ถูก หลอกลวง.',
 'ฤดู ร้อน ใน กวางตุ้ง ค่อนข้าง ร้อน.',
 'ฉัน ชอบ ดู เกม วอลเลย

In [ ]:
textfile = open("/content/T5_en2th", "w")

for element in thai_preds:

    textfile.write(element + "\n")

textfile.close()

In [ ]:
eng_th_bleu = sacrebleu.corpus_bleu(thai_preds, thai_truth)
print("English to Thai: ", eng_th_bleu.score)

English to Thai:  0.1010877860789219
